In [ ]:
import sys
import json
import random
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import napari
from scipy.interpolate import griddata
from pathlib import Path


project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

sys.path.append(f"{project_root}src/")
from utilities import KernelNN, KernelNN_subgraph, GraphLMDBReader, setup_device, get_output_affinities

sys.path.append(f"{project_root}/modis_himalaya/scripts/")
from utilities_modis_himalaya import MODIS_Himalaya_Graph_Generator as Data_Generator
data_path = project_root / 'modis_himalaya' / 'data' / 'himalaya'

device = setup_device()
print(f'Using device: {device}')

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import subprocess

def load_model(id:str, model_type:str='random_crop'):
    model_path = project_root / 'modis_himalaya' / 'trained_models' / model_type / f'{id}.pth'
    config_path = project_root / 'modis_himalaya' / 'trained_models' / model_type / 'logs' / 'configs' / f'{id}.json' 
    with open(config_path, 'r') as f:
        config = json.load(f)
    model = KernelNN(
        width=config['width'],
        ker_width=config['ker_width'],
        depth=config['depth'],
        ker_in=config['edge_attr'],
        in_width=config['node_features'],
        out_width=config['out_width']
    ).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    return model, config

In [ ]:
model, config = load_model(id='testtraining2_2026-05-05_t08-38-27', model_type='regular')

In [ ]:
bbox = config["bbox"]
k = config["k"]
perplexity = config["perplexity"]
sampling_method = config["sampling_method"]
feature_mode = config["feature_mode"]
min_file_size_mb = config.get("min_file_size_mb", 150)
print(f'bbox: {bbox}')
print(f'node_features: {config["node_features"]}')

In [ ]:
Generator = Data_Generator(data_path)

In [ ]:
# Get available timestamps, sort by file size, select N largest
Generator = Data_Generator(data_path)
all_timestamps = Generator.get_available_timestamps(
    min_date='2018-01-01',
    max_date='2018-12-31',
    min_file_size_mb=min_file_size_mb
)
timestamp_info = []
for ts in all_timestamps:
    bright_path = os.path.join(Generator.directory, f'brighttemp_1KM_grid_modis_{ts}.tif')
    size = os.path.getsize(bright_path)
    timestamp_info.append((ts, size))
timestamp_info.sort(key=lambda x: x[1], reverse=True)
N_SAMPLES = 30
dates = [ts for ts, _ in timestamp_info[:N_SAMPLES]]
print(f'Selected {len(dates)} timestamps from {len(all_timestamps)} available')
print(f'Date range: {dates[0][:8]} to {dates[-1][:8]}')


bbox = config["bbox"]#[400, 700, 1500, 1800]
N=1000
print('generate data')
graphs=[]
locations_list=[]
X_list=[]
for date in dates:
    Generator.load_data(timestamp=date, bbox=bbox)
    print(f'Total points available: {Generator.da.shape[1]*Generator.da.shape[2]}')
    graph = Generator.sample_graph(N=N, k=k, perplexity=perplexity, sampling_method=sampling_method, gen_feature_mode=feature_mode)
    sample = graph.to(device)
    graphs.append(sample)
    locations_list.append(pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]}))
    X_list.append(sample.normed_values.detach().cpu().numpy())

# Predict (NOFE, PCA, t-SNE, UMAP)

In [ ]:
pred_nofe_list = []
pred_pca_list = []
pred_tsne_list = []
pred_umap_list = []
for sample in graphs:
    print('NOFE ...', end='\r')
    pred_nofe = model(sample).detach().cpu().numpy()
    pred_nofe_list.append(pred_nofe)
    X = sample.normed_values.detach().cpu().numpy()
    print('PCA ...', end='\r')
    pca = PCA(n_components=3)
    pred_pca = pca.fit_transform(X)
    pred_pca_list.append(pred_pca)
    print('t-SNE ...', end='\r')
    pred_tsne = TSNE(n_components=3, perplexity=30, learning_rate="auto", random_state=42).fit_transform(X)
    pred_tsne_list.append(pred_tsne)
    print('UMAP ...', end='\r')
    umap_id = 'umap_buffer'
    dir = f'{project_root}/ERA5/umap'
    os.makedirs(dir, exist_ok=True)
    np.save(f'{dir}/{umap_id}.npy', X)
    subprocess.run(f"cd {dir} && conda run -n umap python execute_umap.py {umap_id}", shell=True)
    pred_umap = np.load(f'{dir}/res_{umap_id}.npy')
    pred_umap_list.append(pred_umap)
    os.remove(f'{dir}/{umap_id}.npy')
    os.remove(f'{dir}/res_{umap_id}.npy')


# Distance Preservation Functions

In [ ]:
def pairwise_distances(X):
    norms = np.sum(X**2, axis=1, keepdims=True)
    dist2 = norms + norms.T - 2 * X @ X.T
    dist2 = np.maximum(dist2, 0.0)
    return np.sqrt(dist2)

def distance_preservation(H, L):
    DH = pairwise_distances(H)
    DL = pairwise_distances(L)
    i = np.triu_indices_from(DH, k=1)
    dH = DH[i]
    dL = DL[i]
    corr = np.corrcoef(dH, dL)[0, 1]
    return corr

def stress(H, L):
    DH = pairwise_distances(H)
    DL = pairwise_distances(L)
    i = np.triu_indices_from(DH, k=1)
    dH = DH[i]
    dL = DL[i]
    stress = np.sqrt(np.sum((dH - dL)**2) / np.sum(dH**2))
    return stress

In [ ]:
def edge_distances(X, edge_indices):
    diff = X[edge_indices[0]] - X[edge_indices[1]]
    return np.linalg.norm(diff, axis=1)

def edge_distance_preservation(H, L, edge_indices):
    dH = edge_distances(H, edge_indices)
    dL = edge_distances(L, edge_indices)
    corr = np.corrcoef(dH, dL)[0, 1]
    return corr

def edge_stress(H, L, edge_indices):
    dH = edge_distances(H, edge_indices)
    dL = edge_distances(L, edge_indices)   
    return np.sqrt(np.sum((dH - dL)**2) / np.sum(dH**2))

# Evaluation Loop

In [ ]:
nofe_preservation, nofe_stress = [], []
nofe_edge_preservation, nofe_edge_stress = [], []

pca_preservation, pca_stress = [], []
pca_edge_preservation, pca_edge_stress = [], []

tsne_preservation, tsne_stress = [], []
tsne_edge_preservation, tsne_edge_stress = [], []

umap_preservation, umap_stress = [], []
umap_edge_preservation, umap_edge_stress = [], []

for i in range(len(graphs)):
    edge_indices = graphs[i].edge_index.detach().cpu().numpy()
    H = graphs[i].normed_values.detach().cpu().numpy()

    Lnofe = pred_nofe_list[i]
    Lpca  = pred_pca_list[i]
    Ltsne = pred_tsne_list[i]
    Lumap = pred_umap_list[i]

    nofe_preservation.append(distance_preservation(H, Lnofe))
    nofe_stress.append(stress(H, Lnofe))
    nofe_edge_preservation.append(edge_distance_preservation(H, Lnofe, edge_indices))
    nofe_edge_stress.append(edge_stress(H, Lnofe, edge_indices))

    pca_preservation.append(distance_preservation(H, Lpca))
    pca_stress.append(stress(H, Lpca))
    pca_edge_preservation.append(edge_distance_preservation(H, Lpca, edge_indices))
    pca_edge_stress.append(edge_stress(H, Lpca, edge_indices))

    tsne_preservation.append(distance_preservation(H, Ltsne))
    tsne_stress.append(stress(H, Ltsne))
    tsne_edge_preservation.append(edge_distance_preservation(H, Ltsne, edge_indices))
    tsne_edge_stress.append(edge_stress(H, Ltsne, edge_indices))

    umap_preservation.append(distance_preservation(H, Lumap))
    umap_stress.append(stress(H, Lumap))
    umap_edge_preservation.append(edge_distance_preservation(H, Lumap, edge_indices))
    umap_edge_stress.append(edge_stress(H, Lumap, edge_indices))


print("NOFE")
print("preservation:", np.mean(nofe_preservation), "+/-", np.std(nofe_preservation))
print("stress:", np.mean(nofe_stress), "+/-", np.std(nofe_stress))
print("edge preservation:", np.mean(nofe_edge_preservation), "+/-", np.std(nofe_edge_preservation))
print("edge stress:", np.mean(nofe_edge_stress), "+/-", np.std(nofe_edge_stress))
print("---")

print("PCA")
print("preservation:", np.mean(pca_preservation), "+/-", np.std(pca_preservation))
print("stress:", np.mean(pca_stress), "+/-", np.std(pca_stress))
print("edge preservation:", np.mean(pca_edge_preservation), "+/-", np.std(pca_edge_preservation))
print("edge stress:", np.mean(pca_edge_stress), "+/-", np.std(pca_edge_stress))
print("---")

print("TSNE")
print("preservation:", np.mean(tsne_preservation), "+/-", np.std(tsne_preservation))
print("stress:", np.mean(tsne_stress), "+/-", np.std(tsne_stress))
print("edge preservation:", np.mean(tsne_edge_preservation), "+/-", np.std(tsne_edge_preservation))
print("edge stress:", np.mean(tsne_edge_stress), "+/-", np.std(tsne_edge_stress))
print("---")

print("UMAP")
print("preservation:", np.mean(umap_preservation), "+/-", np.std(umap_preservation))
print("stress:", np.mean(umap_stress), "+/-", np.std(umap_stress))
print("edge preservation:", np.mean(umap_edge_preservation), "+/-", np.std(umap_edge_preservation))
print("edge stress:", np.mean(umap_edge_stress), "+/-", np.std(umap_edge_stress))

In [ ]:

methods = {
    "NOFE": {
        "preservation": nofe_preservation,
        "stress": nofe_stress,
        "edge_stress": nofe_edge_stress
    },
    "PCA": {
        "preservation": pca_preservation,
        "stress": pca_stress,
        "edge_stress": pca_edge_stress
    },
    "TSNE": {
        "preservation": tsne_preservation,
        "stress": tsne_stress,
        "edge_stress": tsne_edge_stress
    },
    "UMAP": {
        "preservation": umap_preservation,
        "stress": umap_stress,
        "edge_stress": umap_edge_stress
    }
}

for method, metrics in methods.items():
    pres_mean = np.mean(metrics["preservation"])
    pres_std  = np.std(metrics["preservation"])

    stress_mean = np.mean(metrics["stress"])
    stress_std  = np.std(metrics["stress"])

    edge_mean = np.mean(metrics["edge_stress"])
    edge_std  = np.std(metrics["edge_stress"])

    print(
        f"{method}: "
        f"{pres_mean:.3f} $\\pm$ {pres_std:.3f} & "
        f"{stress_mean:.3f} $\\pm$ {stress_std:.3f} & "
        f"{edge_mean:.3f} $\\pm$ {edge_std:.3f}"
    )